# Exercise 6: The two emittances

**Block 2: Twiss, ellipses and emittance**

**Learning objectives**
* Generate a Gaussian beam from its $\Sigma$ (sigma) matrix and compute the **rms emittance** $\varepsilon_\text{rms}=\sqrt{\text{det }\Sigma}$.
* Compute each particle's **single-particle emittance** $\varepsilon_i$ and show that for a matched beam it is
  exponentially distributed with mean $2\varepsilon_\text{rms}$.
* Understand why the $1\sigma$ ellipse contains only 39 % of the particles (not 68 %).
* See what happens to both quantities for a **mismatched** beam.

**Motivation.** The word "emittance" is used for two different things, and mixing them up is one
of the most common sources of confusion in accelerator physics. This exercise keeps them apart.

> **Student version.** Fill in the cells marked `# YOUR CODE HERE` and run the check cells after them: they tell you when you are right.

## The two emittances

| | single-particle emittance $\varepsilon_i$ | rms (beam) emittance $\varepsilon_\text{rms}$ |
|---|---|---|
| belongs to | **one particle** | **the whole beam** (a distribution) |
| definition | $\varepsilon_i = \gamma x^2 + 2\alpha x p_x + \beta p_x^2$ | $\varepsilon_\text{rms} = \sqrt{\text{det} \Sigma}$, with $\Sigma$ = [[⟨$x^2$⟩, ⟨$x px$⟩], [⟨$x p_x$⟩, ⟨$p_x^2$⟩]] |
| meaning | area/$\pi$ of the ellipse this particle moves on | area/$\pi$ of the beam's "$1\sigma$ ellipse" |
| needs the lattice ($\beta$, $\alpha$)? | **yes** | **no**, only the particle coordinates |

⟨…⟩ is the average over all particles (after subtracting the mean). We also use the **action**
$J_i = \varepsilon_i/2$, which is common in the literature.

Units: 1 mm mrad = $10^{-6}$ m rad, the rad is often omitted and we write 1 $\mu$m.

In [ ]:
# --- setup: make `beamlib` importable --------------------------------------
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "beamlib").is_dir())
sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from beamlib import elements as E, optics as O, lattices as L, tracking as T, diagnostics as D, plotting as P
from beamlib.plotting import COLORS, MM
P.setup()
rng = np.random.default_rng(2024)

# We observe the beam at location B of Exercise 5 (entrance of the first dipole), where alpha != 0.
ring = L.fodo_ring()
ringB = ring[2:] + ring[:2]
M_B = O.one_turn_matrix(ringB)[:4, :4]          # linear ring: the one-turn matrix is all we need
one_turn = E.LinearMap(M_B).track
tw = O.twiss_from_matrix(M_B[:2, :2])
beta, alpha, gamma = tw["beta"], tw["alpha"], tw["gamma"]
print(f"Twiss at B: beta = {beta:.3f} m, alpha = {alpha:.3f}, gamma = {gamma:.4f} 1/m, Q = {tw['Q']:.4f}")
eps = 1e-6        # 1 mm mrad

## 1. A Gaussian beam from its Σ matrix

A beam **matched** to the lattice has a Σ matrix with the same shape as the lattice ellipse:

$$\Sigma = \varepsilon \begin{pmatrix} \beta & -\alpha \\ -\alpha & \gamma \end{pmatrix}
\quad\Rightarrow\quad \sqrt{\det\Sigma} = \varepsilon\sqrt{\beta\gamma - \alpha^2} = \varepsilon .$$

To draw particles with a given $\Sigma$: take independent unit Gaussians $u$ (`rng.standard_normal`) and
multiply by the Cholesky factor $L$ of $\Sigma$ ($\Sigma = L L^T$, `np.linalg.cholesky`). Then $L u$ has covariance $\Sigma$.

**Task.** Implement `generate_beam` and `rms_emittance`. Use `np.cov(x, px, bias=True)` or compute the
averages yourself.

In [ ]:
def generate_beam(Sigma: np.ndarray, n: int, rng) -> np.ndarray:
    """n particles (array (2, n): rows x, px) from a Gaussian with covariance Sigma."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

def rms_emittance(x: np.ndarray, px: np.ndarray) -> float:
    """sqrt(det Sigma) with centred second moments."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
Sigma_matched = eps * np.array([[beta, -alpha], [-alpha, gamma]])
N = 100_000
beam = generate_beam(Sigma_matched, N, rng)
x, px = beam[0, :], beam[1, :]
eps_rms = rms_emittance(x, px)
print(f"sample covariance / Sigma =\n{(np.cov(x, px, bias=True) / Sigma_matched).round(3)}")
print(f"eps_rms = {eps_rms * 1e6:.4f} mm mrad")
assert np.allclose(np.cov(x, px, bias=True), Sigma_matched, rtol=0.03), "covariance should match Sigma"
assert abs(eps_rms / eps - 1) < 0.01
assert np.isclose(eps_rms, D.rms_emittance(x, px))
print("GOOD! Beam generated with the right Sigma and eps_rms")

## 2. The distribution of single-particle emittances

**Predict first.** The beam has $\varepsilon_\text{rms}$ = 1 mm mrad. What do you expect the *average*
single-particle emittance ⟨$\varepsilon_i$⟩ to be: smaller than, equal to, or larger than 1 mm mrad?
What shape will the histogram of $\varepsilon_i$ have?

*Your prediction:*

**Why.** In normalised coordinates ($\hat{x}, $\hat{p}_x$) (next exercise) a matched beam is a *round* Gaussian
with variance $\varepsilon_\text{rms}$ in each direction, and $\varepsilon_i = \hat{x}^2 + \hat{p}_x^2$ is the squared distance from the
centre. The sum of two squared Gaussians, each with variance $\varepsilon_\text{rms}$, follows an
**exponential distribution with mean $2\varepsilon_\text{rms}$**:

$$P(\varepsilon_i) = \frac{1}{2\varepsilon_{rms}}\, e^{-\varepsilon_i / 2\varepsilon_{rms}} .$$

So ⟨$\varepsilon_i$⟩ = $2 \varepsilon_\text{rms}$ and ⟨$J_i$⟩ = $\varepsilon_\text{rms}$.

In [ ]:
eps_i = D.cs_invariant(x, px, beta, alpha)
print(f"<eps_i> = {eps_i.mean() * 1e6:.4f} mm mrad,  2 eps_rms = {2 * eps_rms * 1e6:.4f} mm mrad")
assert abs(eps_i.mean() / (2 * eps_rms) - 1) < 0.01

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0, 12, 61)
axs[0].hist(eps_i * 1e6, bins=bins, density=True, color=COLORS["x"], alpha=0.6, label="particles")
e = np.linspace(0, 12, 300)
axs[0].plot(e, np.exp(-e / 2) / 2, color=COLORS["theory"], label=r"$e^{-\varepsilon_i/2\varepsilon_\text{rms}} / 2\varepsilon_\text{rms}$")
axs[0].axvline(2 * eps_rms * 1e6, color=COLORS["accent"], ls="--", label=r"mean = $2\varepsilon_\text{rms}$")
axs[0].set_xlabel(r"$\varepsilon_i$ [mm mrad]"); axs[0].set_ylabel("probability density"); axs[0].legend()
axs[1].hist(eps_i * 1e6, bins=bins, density=True, color=COLORS["x"], alpha=0.6)
axs[1].plot(e, np.exp(-e / 2) / 2, color=COLORS["theory"])
axs[1].set_yscale("log"); axs[1].set_xlabel(r"$\varepsilon_i$ [mm mrad]")
axs[1].set_title("log scale: an exponential is a straight line")
plt.show()

## 3. How many particles are inside the $1\sigma$ ellipse?

The **$1\sigma$ ellipse** of the beam is the lattice ellipse with $\varepsilon_i = \varepsilon_\text{rms}$. Its half-width in
$x$ is exactly $\sigma_x = \sqrt{\beta \varepsilon_\text{rms}}$.

**Task.** Count the fraction of particles with $\varepsilon_i \leq \varepsilon_\text{rms}$ (inside the $1\sigma$ ellipse), and the
fraction with $|x| \leq \sigma_x$ (inside the $1\sigma$ *band* in $x$ only). Compare with $1 − e^{-1/2}$ and with 68.3 %.

In [ ]:
# YOUR CODE HERE
raise NotImplementedError("Replace this line with your code")
print(f"inside the 1 sigma ellipse: {frac_ellipse * 100:.2f} %   (theory 1 - exp(-1/2) = {(1 - np.exp(-0.5)) * 100:.2f} %)")
print(f"inside |x| < sigma_x      : {frac_band * 100:.2f} %   (theory 68.27 %)")
assert abs(frac_ellipse - (1 - np.exp(-0.5))) < 0.005
assert abs(frac_band - 0.6827) < 0.005
print("GOOD! 39 % in the 2D ellipse, 68 % in the 1D band")
for k in (1, 2, 3):
    print(f"  {k} sigma ellipse: measured {np.mean(eps_i <= k**2 * eps_rms) * 100:5.1f} %, "
          f"theory 1 - exp(-{k}^2/2) = {(1 - np.exp(-k**2 / 2)) * 100:5.1f} %")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
inside = eps_i <= eps_rms
ax.scatter(x[~inside][:6000] * MM, px[~inside][:6000] * MM, s=1, color=COLORS["grey"], label="outside")
ax.scatter(x[inside][:4000] * MM, px[inside][:4000] * MM, s=1, color=COLORS["x"], label=r"inside $1\sigma$ ellipse")
P.plot_ellipse(ax, eps_rms, beta, alpha, label=r"$1\sigma$ ellipse")
P.plot_ellipse(ax, 4 * eps_rms, beta, alpha, ls="--", label=r"$2\sigma$ ellipse")
sx = np.sqrt(beta * eps_rms)
ax.axvspan(-sx * MM, sx * MM, color=COLORS["accent"], alpha=0.1, label=r"$|x| < \sigma_x$ band")
ax.set_xlabel(r"$x$ [mm]"); ax.set_ylabel(r"$p_x$ [mrad]"); ax.legend(loc="upper right", markerscale=5)
plt.show()

The 68 % rule is for a **one-dimensional** Gaussian. The ellipse is a cut in **two** dimensions:
a particle must be "within $1\sigma$" in both directions at once, which is much rarer. The band
$|x| < \sigma_x$ contains many particles with large $|p_x|$ that are outside the ellipse.

## 4. A mismatched beam

Now the beam does *not* have the lattice's shape. We take a beam whose "$\beta$" is 2.5 times larger
than the lattice $\beta$, with the same $\alpha$ (this happens for example when the injection optics is wrong):

$$\Sigma_b = \varepsilon \begin{pmatrix} \beta_b & -\alpha_b \\ -\alpha_b & \gamma_b \end{pmatrix},\quad \beta_b = 2.5\,\beta,\ \alpha_b = \alpha .$$

Its rms emittance is still $\sqrt{\text{det} \Sigma_b} = \varepsilon$. But the particles do not sit on lattice ellipses
in a "balanced" way. Averaging $\varepsilon_i$ over this beam gives

$$\frac{\langle\varepsilon_i\rangle}{2} = \varepsilon_{rms}\cdot B, \qquad
B = \tfrac12\left(\beta\gamma_b - 2\alpha\alpha_b + \gamma\beta_b\right) \ \ge 1,$$

with $B$ = 1 only if ($\beta_b$, $\alpha_b$) = ($\beta$, $\alpha$).

**Task.** Implement the mismatch factor $B$ and check it against the beam.

In [ ]:
def mismatch_factor(beta, alpha, beta_b, alpha_b) -> float:
    """B = (beta gamma_b - 2 alpha alpha_b + gamma beta_b) / 2  (>= 1)."""
    # YOUR CODE HERE
    raise NotImplementedError("Replace this line with your code")

In [ ]:
beta_b, alpha_b = 2.5 * beta, alpha
gamma_b = (1 + alpha_b**2) / beta_b
Sigma_mis = eps * np.array([[beta_b, -alpha_b], [-alpha_b, gamma_b]])
beam_mis = np.zeros((4, N))
beam_mis[:2, :] = generate_beam(Sigma_mis, N, rng)

Bmag = mismatch_factor(beta, alpha, beta_b, alpha_b)
half_mean = 0.5 * D.cs_invariant(beam_mis[0, :], beam_mis[1, :], beta, alpha).mean()
erms_mis = rms_emittance(beam_mis[0, :], beam_mis[1, :])
print(f"mismatch factor B = {Bmag:.4f}")
print(f"eps_rms = sqrt(det Sigma) = {erms_mis * 1e6:.4f} mm mrad")
print(f"<eps_i>/2                 = {half_mean * 1e6:.4f} mm mrad  (B * eps_rms = {Bmag * erms_mis * 1e6:.4f})")
assert Bmag > 1 and abs(half_mean / (Bmag * erms_mis) - 1) < 0.01
assert np.isclose(mismatch_factor(beta, alpha, beta, alpha), 1.0), "a matched beam must give B = 1"
print("GOOD! <eps_i>/2 >= eps_rms, with equality only for a matched beam")

**Predict first.** We now track the mismatched beam for 60 turns. Which of these stay constant:
(a) the beam size $\sigma_x$, (b) $\epsilon_\text{rms} = \sqrt{\text{det} \Sigma}$, (c) ⟨$\epsilon_i$⟩/2?

*Your prediction:*

In [ ]:
n_track = 20_000                      # a subset is enough (and keeps memory small)
res = T.track(beam_mis[:, :n_track], one_turn, 60)
res_m = T.track(np.vstack([beam, np.zeros((2, N))])[:, :n_track], one_turn, 60)
sig_x = np.std(res.x, axis=1)
e_rms_t = D.rms_emittance(res.x, res.px)
half_t = 0.5 * D.cs_invariant(res.x, res.px, beta, alpha).mean(axis=1)

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(sig_x * MM, "o-", ms=3, color=COLORS["x"], label="mismatched beam")
axs[0].plot(np.std(res_m.x, axis=1) * MM, "o-", ms=3, color=COLORS["grey"], label="matched beam")
axs[0].set_xlabel("turn"); axs[0].set_ylabel(r"$\sigma_x$ [mm]"); axs[0].legend()
axs[0].set_title("beam size at B, turn by turn")
axs[1].plot(e_rms_t * 1e6, color=COLORS["x"], label=r"$\varepsilon_\text{rms}=\sqrt{\det\Sigma}$")
axs[1].plot(half_t * 1e6, color=COLORS["accent"], label=r"$\langle\varepsilon_i\rangle / 2$")
axs[1].set_ylim(0, 1.3 * half_t.max() * 1e6); axs[1].set_xlabel("turn"); axs[1].set_ylabel("mm mrad"); axs[1].legend()
axs[1].set_title("both emittances are constant in a linear ring")
plt.show()
assert np.allclose(e_rms_t, e_rms_t[0], rtol=1e-9) and np.allclose(half_t, half_t[0], rtol=1e-9)

The beam size *beats* turn after turn (at twice the tune: the beam ellipse rotates inside the lattice
ellipse and looks the same after half a rotation). Both emittances stay constant, but they are
**different** numbers.

In normalised coordinates (Exercise 7) the picture is clearest: the lattice ellipses become circles,
the matched beam is round, and the mismatched beam is an ellipse that turns by the phase advance $\mu$
every turn.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4), sharex=True, sharey=True)
for ax, turn in zip(axs, [0, 1, 2, 3]):
    P.phase_space(ax, res.x[turn, :3000], res.px[turn, :3000], beta, alpha, s=1, color=COLORS["x"])
    P.plot_ellipse(ax, eps, beta, alpha, normalised=True, color="k", ls="--")
    ax.set_title(f"turn {turn}")
fig.suptitle("Mismatched beam in normalised coordinates (dashed: matched 1σ circle)")
plt.show()

## Summary

| | $\varepsilon_\text{rms} = \sqrt{\det \Sigma}$ | ⟨$\varepsilon_i$⟩ / 2 = ⟨$J$⟩ |
|---|---|---|
| depends on | the particle distribution only | the distribution **and** the lattice optics ($\beta$, $\alpha$) |
| matched Gaussian beam | $\varepsilon$ | $\varepsilon$ |
| mismatched beam | $\varepsilon$ | $B \varepsilon > \varepsilon$ |
| linear ring | constant | constant |
| ring with nonlinearities (Exercise 15) | **grows** as the beam filaments, towards ⟨$\varepsilon_i$⟩/2 | stays (nearly) constant |

The second column tells you the emittance the beam *will* have once the lattice has "mixed" it:
the lattice only respects the particles' own invariants $\varepsilon_i$. This is why injection mismatch leads to
emittance growth in real machines.

## Reflection questions

1. A colleague says "the beam emittance is 1 mm mrad, so every particle has $\varepsilon_i$ = 1 mm mrad".
   What is wrong with this statement?
2. Why does the $1\sigma$ ellipse contain 39 % of the particles while |$x$| < $\sigma_x$ contains 68 %?
3. Can you measure $\varepsilon_\text{rms}$ without knowing the lattice? Can you measure ⟨$\varepsilon_i$⟩?